In [ ]:
import squigglepy as sq
import numpy as np
import pandas as pd
from squigglepy.numbers import K, M, B

sq.set_seed(42)
np.random.seed(42)
np.seterr(invalid='raise')
N_SAMPLES = 5000

from chip_estimates_utils import estimate_cumulative_chip_sales

In [ ]:
CHIP_TYPES = ['A100', 'A800', 'H100/H200', 'H800', 'H20', 'B200', 'B300']
HARDWARE_SHARE = sq.to(0.96, 0.99, credibility=80)

revenue_df = pd.read_csv(
    "https://docs.google.com/spreadsheets/d/1Yhu87Rw--9tviAuBwg_luL3OFAFkdHdVfli6tN215Xk/export?format=csv&gid=0"
).set_index('Quarter')

prices_df = pd.read_csv(
    "https://docs.google.com/spreadsheets/d/1Yhu87Rw--9tviAuBwg_luL3OFAFkdHdVfli6tN215Xk/export?format=csv&gid=1819303346"
).set_index('Year')

QUARTERS = revenue_df.index.tolist()

In [ ]:
import requests
import zipfile
import io

CHIP_NAME_MAP = {
    'A100': 'A100', 'A800': 'A800', 'H100/H200': 'H100',
    'H800': 'H800', 'H20': 'H20', 'B200': 'B200', 'B300': 'B300',
}

FALLBACK_SPECS = {
    'A100':      {'tops': 624,  'tdp': 400},
    'A800':      {'tops': 312,  'tdp': 400},
    'H100/H200': {'tops': 1979, 'tdp': 700},
    'H800':      {'tops': 1979, 'tdp': 700},
    'H20':       {'tops': 296,  'tdp': 400},
    'B200':      {'tops': 5000, 'tdp': 1200},
    'B300':      {'tops': 5000, 'tdp': 1400},
}

H100_TOPS = 1979

def load_chip_specs(chip_types_df, chip_name_map, fallback_specs):
    specs = {}
    for notebook_name, csv_name in chip_name_map.items():
        row = chip_types_df[chip_types_df['Name'] == csv_name]
        if len(row) == 1:
            tops_raw = row['8-bit OP/s'].values[0]
            tops = float(tops_raw) / 1e12 if pd.notna(tops_raw) else fallback_specs[notebook_name]['tops']
            tdp_col = 'TDP (W) (from ML Hardware (linked))'
            tdp_raw = row[tdp_col].values[0] if tdp_col in row.columns else None
            tdp = float(tdp_raw) if pd.notna(tdp_raw) else fallback_specs[notebook_name]['tdp']
            specs[notebook_name] = {'tops': tops, 'tdp': tdp}
        else:
            specs[notebook_name] = fallback_specs[notebook_name].copy()
    return specs

try:
    response = requests.get("https://epoch.ai/data/ai_chip_sales.zip", timeout=10)
    response.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        with z.open("chip_types.csv") as f:
            chip_types_df = pd.read_csv(f)
            nvidia_chips_df = chip_types_df[chip_types_df['Designer'] == 'Nvidia']
    CHIP_SPECS = load_chip_specs(nvidia_chips_df, CHIP_NAME_MAP, FALLBACK_SPECS)
except Exception as e:
    print(f"Warning: Could not download chip specs: {e}. Using fallbacks.")
    CHIP_SPECS = FALLBACK_SPECS.copy()

for chip, spec in CHIP_SPECS.items():
    print(f"  {chip}: TOPS={spec['tops']:.0f}, TDP={spec['tdp']:.0f}W")

In [ ]:
# ==============================================
# PRICES AND DEFLATION
# ==============================================

PRICE_COLUMN_MAP = {'H100/H200': 'H100'}

FALLBACK_PRICES = {
    'A100': (10*K, 15*K), 'A800': (10*K, 15*K), 'H100/H200': (20*K, 30*K),
    'H800': (20*K, 30*K), 'H20': (10*K, 15*K), 'B200': (33*K, 42*K), 'B300': (33*K, 42*K)
}

def get_price_dist_for_year(chip, year):
    csv_chip_name = PRICE_COLUMN_MAP.get(chip, chip)
    low_col, high_col = f'{csv_chip_name} low', f'{csv_chip_name} high'
    if low_col in prices_df.columns and high_col in prices_df.columns:
        if year in prices_df.index:
            low = prices_df.loc[year, low_col]
            high = prices_df.loc[year, high_col]
            if pd.notna(low) and pd.notna(high):
                return sq.to(low, high, credibility=80)
    return sq.to(*FALLBACK_PRICES.get(chip, (20*K, 30*K)), credibility=80)

def find_first_year_with_price(chip):
    csv_chip_name = PRICE_COLUMN_MAP.get(chip, chip)
    low_col = f'{csv_chip_name} low'
    if low_col in prices_df.columns:
        for year in sorted(prices_df.index):
            if pd.notna(prices_df.loc[year, low_col]):
                return year
    return min(prices_df.index)

BASE_YEAR = {chip: find_first_year_with_price(chip) for chip in CHIP_TYPES}
BASE_PRICES = {chip: get_price_dist_for_year(chip, BASE_YEAR[chip]) for chip in CHIP_TYPES}

def get_price_bounds(chip, year):
    csv_chip_name = PRICE_COLUMN_MAP.get(chip, chip)
    low_col, high_col = f'{csv_chip_name} low', f'{csv_chip_name} high'
    if low_col in prices_df.columns and high_col in prices_df.columns:
        if year in prices_df.index:
            low = prices_df.loc[year, low_col]
            high = prices_df.loc[year, high_col]
            if pd.notna(low) and pd.notna(high):
                return (low, high)
    return None

def get_price_year_for_quarter(quarter):
    start_date = revenue_df.loc[quarter, 'Start Date']
    return pd.to_datetime(start_date).year

def get_deflation_factor(quarter, chip):
    price_year = get_price_year_for_quarter(quarter)
    base_year = BASE_YEAR[chip]
    if price_year <= base_year:
        return 1.0
    base_bounds = get_price_bounds(chip, base_year)
    current_bounds = get_price_bounds(chip, price_year)
    if base_bounds and current_bounds:
        return np.sqrt((current_bounds[0] * current_bounds[1]) / (base_bounds[0] * base_bounds[1]))
    return 1.0

The next cell runs a Monte Carlo simulation to estimate chip volumes over time by dividing revenue among chip types (based on production mix) and then dividing by chip price.

Chip prices for each chip model are presampled once and reused across every quarter. This ensures that uncertainty is applied across quarters; otherwise, sampling these parameters separately across every quarter means that much of the uncertainty over the cumulative total gets canceled out. Because the samples are sorted (e.g. the first chip count in each quarter's sample represents is based on the same price draw), the sum of samples across quarters models uncertainty in the total chips delivered across quarters. Base prices are also partially correlated across chip type; this similarly widens the confidence intervals on aggregate totals.

The results are stored in the form of arrays of samples of chip counts, keyed by chip type and quarter. We then use percentiles to find medians and confidence intervals.

In [ ]:
# ==============================================
# SAMPLING FUNCTIONS AND BASELINE SIMULATION
# ==============================================

def sample_revenue(quarter):
    return revenue_df.loc[quarter, 'Compute revenue'] * B

def sample_shares(quarter):
    return {chip: revenue_df.loc[quarter, f'{chip} share'] for chip in CHIP_TYPES}

def sample_base_price(chip):
    return BASE_PRICES[chip] @ 1

def sample_revenue_uncertainty():
    return HARDWARE_SHARE @ 1

quarterly_samples = estimate_cumulative_chip_sales(
    quarters=QUARTERS,
    chip_types=CHIP_TYPES,
    sample_revenue=sample_revenue,
    sample_shares=sample_shares,
    sample_base_price=sample_base_price,
    get_deflation_factor=get_deflation_factor,
    sample_revenue_uncertainty=sample_revenue_uncertainty,
    n_samples=N_SAMPLES
)

print(f"Baseline simulation complete: {len(QUARTERS)} quarters, {N_SAMPLES} samples")

## Counterfactual Analysis: What if newer GPUs hadn't been introduced?

Two scenarios, keeping total dollar revenue the same but changing what chips that money buys:

1. **All → A100**: Reallocate all H100/H200, H800, B200, and B300 revenue to A100 chips (H20 left unchanged since it's a nerfed export chip). Uses 2025 A100 pricing ($10K–$15K).

2. **Blackwell → H100**: Reallocate only B200 and B300 revenue to H100/H200 chips (everything else unchanged). Uses 2025 H100 pricing ($22K–$30K).

The key question: how much does each GPU generation upgrade actually contribute to cumulative compute, given that newer chips cost more per unit but deliver more TOPS?

In [ ]:
# ==============================================
# COUNTERFACTUAL SIMULATIONS
# ==============================================

# --- Scenario 1: All H100/H200, H800, B200, B300 → A100 ---
# Same dollar revenue, but buying A100s instead of newer chips

REPLACED_BY_A100 = ['H100/H200', 'H800', 'B200', 'B300']
A100_PRICE_2025 = get_price_dist_for_year('A100', 2025)

def sample_base_price_all_a100(chip):
    if chip in REPLACED_BY_A100:
        return A100_PRICE_2025 @ 1
    return BASE_PRICES[chip] @ 1

def get_deflation_factor_all_a100(quarter, chip):
    if chip in REPLACED_BY_A100:
        return 1.0  # Fixed 2025 price, no deflation
    return get_deflation_factor(quarter, chip)

# Replaced chips produce A100-level compute
CHIP_SPECS_ALL_A100 = {}
for chip in CHIP_TYPES:
    if chip in REPLACED_BY_A100:
        CHIP_SPECS_ALL_A100[chip] = CHIP_SPECS['A100'].copy()
    else:
        CHIP_SPECS_ALL_A100[chip] = CHIP_SPECS[chip]

cf1_quarterly = estimate_cumulative_chip_sales(
    quarters=QUARTERS, chip_types=CHIP_TYPES,
    sample_revenue=sample_revenue, sample_shares=sample_shares,
    sample_base_price=sample_base_price_all_a100,
    get_deflation_factor=get_deflation_factor_all_a100,
    sample_revenue_uncertainty=sample_revenue_uncertainty,
    n_samples=N_SAMPLES
)

# --- Scenario 2: Blackwell (B200, B300) → H100/H200 ---
# Same dollar revenue, but buying H100s instead of Blackwell chips

REPLACED_BY_H100 = ['B200', 'B300']
H100_PRICE_2025 = get_price_dist_for_year('H100/H200', 2025)

def sample_base_price_no_blackwell(chip):
    if chip in REPLACED_BY_H100:
        return H100_PRICE_2025 @ 1
    return BASE_PRICES[chip] @ 1

def get_deflation_factor_no_blackwell(quarter, chip):
    if chip in REPLACED_BY_H100:
        return 1.0  # Fixed 2025 price, no deflation
    return get_deflation_factor(quarter, chip)

# Replaced chips produce H100-level compute
CHIP_SPECS_NO_BLACKWELL = {}
for chip in CHIP_TYPES:
    if chip in REPLACED_BY_H100:
        CHIP_SPECS_NO_BLACKWELL[chip] = CHIP_SPECS['H100/H200'].copy()
    else:
        CHIP_SPECS_NO_BLACKWELL[chip] = CHIP_SPECS[chip]

cf2_quarterly = estimate_cumulative_chip_sales(
    quarters=QUARTERS, chip_types=CHIP_TYPES,
    sample_revenue=sample_revenue, sample_shares=sample_shares,
    sample_base_price=sample_base_price_no_blackwell,
    get_deflation_factor=get_deflation_factor_no_blackwell,
    sample_revenue_uncertainty=sample_revenue_uncertainty,
    n_samples=N_SAMPLES
)

# --- Compute fiscal quarter H100e flow and running totals for all scenarios ---

def compute_fiscal_h100e(quarterly_samples, chip_specs, quarters, chip_types, n_samples):
    """Compute per-quarter H100e flow and cumulative running totals (fiscal quarters)."""
    flow = {}
    running = {}
    cumulative = np.zeros(n_samples)
    for quarter in quarters:
        q_flow = np.zeros(n_samples)
        for chip in chip_types:
            q_flow += quarterly_samples[quarter][chip] * (chip_specs[chip]['tops'] / H100_TOPS)
        flow[quarter] = q_flow
        cumulative = cumulative + q_flow
        running[quarter] = cumulative.copy()
    return flow, running

baseline_flow, baseline_running = compute_fiscal_h100e(quarterly_samples, CHIP_SPECS, QUARTERS, CHIP_TYPES, N_SAMPLES)
cf1_flow, cf1_running = compute_fiscal_h100e(cf1_quarterly, CHIP_SPECS_ALL_A100, QUARTERS, CHIP_TYPES, N_SAMPLES)
cf2_flow, cf2_running = compute_fiscal_h100e(cf2_quarterly, CHIP_SPECS_NO_BLACKWELL, QUARTERS, CHIP_TYPES, N_SAMPLES)

# --- Compute CAGR of H100e flow for each scenario ---

def compute_h100e_flow_cagr(flow, quarters, n_samples):
    """Compute average QoQ growth rate of H100e flow (CAGR)."""
    n_transitions = len(quarters) - 1
    first = flow[quarters[0]]
    last = flow[quarters[-1]]
    valid = first > 0
    cagr = np.full(n_samples, np.nan)
    cagr[valid] = (last[valid] / first[valid]) ** (1 / n_transitions) - 1
    return cagr

baseline_cagr = compute_h100e_flow_cagr(baseline_flow, QUARTERS, N_SAMPLES)
cf1_cagr = compute_h100e_flow_cagr(cf1_flow, QUARTERS, N_SAMPLES)
cf2_cagr = compute_h100e_flow_cagr(cf2_flow, QUARTERS, N_SAMPLES)

# Print summaries
last_q = QUARTERS[-1]
print(f"Cumulative H100e at {last_q} (fiscal):")
for label, running in [("Baseline", baseline_running),
                        ("All \u2192 A100", cf1_running),
                        ("Blackwell \u2192 H100", cf2_running)]:
    arr = running[last_q]
    p5, p50, p95 = [np.percentile(arr, p) / 1e6 for p in [5, 50, 95]]
    print(f"  {label:<25} {p50:>6.2f}M  (90% CI: {p5:.2f}M \u2013 {p95:.2f}M)")

b_med = np.percentile(baseline_running[last_q], 50)
c1_med = np.percentile(cf1_running[last_q], 50)
c2_med = np.percentile(cf2_running[last_q], 50)
print(f"\n  All \u2192 A100: {c1_med/b_med:.1%} of baseline")
print(f"  Blackwell \u2192 H100: {c2_med/b_med:.1%} of baseline")

n_trans = len(QUARTERS) - 1
print(f"\nH100e flow CAGR ({QUARTERS[0]} to {QUARTERS[-1]}, {n_trans} quarters):")
for label, cagr in [("Baseline", baseline_cagr),
                     ("All \u2192 A100", cf1_cagr),
                     ("Blackwell \u2192 H100", cf2_cagr)]:
    p5, p50, p95 = np.nanpercentile(cagr, [5, 50, 95])
    a5, a50, a95 = [(1 + g) ** 4 for g in [p5, p50, p95]]
    print(f"  {label:<25} QoQ: {p50:.1%} ({p5:.1%} \u2013 {p95:.1%})  |  Annual: {a50:.2f}x ({a5:.2f}x \u2013 {a95:.2f}x)")

In [ ]:
# ==============================================
# COUNTERFACTUAL VISUALIZATION (FISCAL QUARTERS)
# ==============================================

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Fiscal quarter end dates for x-axis
fiscal_dates = [pd.to_datetime(revenue_df.loc[q, 'End Date']) for q in QUARTERS]

def extract_running_h100e(running, quarters):
    """Extract p5/p50/p95 cumulative H100e in millions."""
    p5 = [np.percentile(running[q], 5) / 1e6 for q in quarters]
    p50 = [np.percentile(running[q], 50) / 1e6 for q in quarters]
    p95 = [np.percentile(running[q], 95) / 1e6 for q in quarters]
    return p5, p50, p95

b = extract_running_h100e(baseline_running, QUARTERS)
c1 = extract_running_h100e(cf1_running, QUARTERS)
c2 = extract_running_h100e(cf2_running, QUARTERS)

# CAGR medians (annualized) for annotation
b_ann = (1 + np.nanpercentile(baseline_cagr, 50)) ** 4
c1_ann = (1 + np.nanpercentile(cf1_cagr, 50)) ** 4
c2_ann = (1 + np.nanpercentile(cf2_cagr, 50)) ** 4

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# --- Left: Baseline vs All \u2192 A100 ---
ax1.fill_between(fiscal_dates, b[0], b[2], alpha=0.15, color='tab:blue')
ax1.plot(fiscal_dates, b[1], 'o-', color='tab:blue',
         label=f'Baseline  ({b_ann:.1f}x/yr flow CAGR)', linewidth=2, markersize=4)
ax1.fill_between(fiscal_dates, c1[0], c1[2], alpha=0.15, color='tab:orange')
ax1.plot(fiscal_dates, c1[1], 's--', color='tab:orange',
         label=f'All \u2192 A100  ({c1_ann:.1f}x/yr flow CAGR)', linewidth=2, markersize=4)

ax1.set_title('What if everything after A100 was just more A100s?', fontsize=12)
ax1.set_ylabel('Cumulative H100-equivalent compute (millions)')
ax1.legend(loc='upper left', fontsize=9)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
ax1.set_xticks(fiscal_dates[::2])
ax1.grid(True, alpha=0.3)

# Annotate final ratio
ratio1 = c1[1][-1] / b[1][-1]
ax1.annotate(f'{ratio1:.0%} of baseline\n({c1[1][-1]:.1f}M vs {b[1][-1]:.1f}M)',
             xy=(fiscal_dates[-1], c1[1][-1]), xytext=(-120, 25),
             textcoords='offset points', fontsize=9, color='tab:orange',
             arrowprops=dict(arrowstyle='->', color='tab:orange', lw=1.5))

# --- Right: Baseline vs Blackwell \u2192 H100 ---
ax2.fill_between(fiscal_dates, b[0], b[2], alpha=0.15, color='tab:blue')
ax2.plot(fiscal_dates, b[1], 'o-', color='tab:blue',
         label=f'Baseline  ({b_ann:.1f}x/yr flow CAGR)', linewidth=2, markersize=4)
ax2.fill_between(fiscal_dates, c2[0], c2[2], alpha=0.15, color='tab:green')
ax2.plot(fiscal_dates, c2[1], 's--', color='tab:green',
         label=f'Blackwell \u2192 H100  ({c2_ann:.1f}x/yr flow CAGR)', linewidth=2, markersize=4)

ax2.set_title('What if Blackwell was just more H100s?', fontsize=12)
ax2.set_ylabel('Cumulative H100-equivalent compute (millions)')
ax2.legend(loc='upper left', fontsize=9)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
ax2.set_xticks(fiscal_dates[::2])
ax2.grid(True, alpha=0.3)

# Annotate final ratio
ratio2 = c2[1][-1] / b[1][-1]
ax2.annotate(f'{ratio2:.0%} of baseline\n({c2[1][-1]:.1f}M vs {b[1][-1]:.1f}M)',
             xy=(fiscal_dates[-1], c2[1][-1]), xytext=(-120, 25),
             textcoords='offset points', fontsize=9, color='tab:green',
             arrowprops=dict(arrowstyle='->', color='tab:green', lw=1.5))

plt.suptitle('Counterfactual: Cumulative Nvidia H100e Compute (Fiscal Quarters)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('visualizations/counterfactual_h100e_cumulative.png', dpi=150, bbox_inches='tight')
plt.show()